In [ ]:
import pandas as pd
import glob
import os
import numpy as np

In [ ]:
input_frame = pd.read_csv("final_reasonable_check_patient_centric_enrollments.csv")

In [ ]:
input_frame.info()

In [ ]:
predictions = input_frame.groupby(['prompt_id'])['eligibility_result'].mean()

In [ ]:
predictions.shape

In [ ]:
all_files = glob.glob(os.path.join('../spaces_for_patients_checks/', "*.csv"))

if not all_files:
    print(f"No CSV files found in the directory: {directory_path}")

list_of_dfs = []
for file_path in all_files:
    try:
        df = pd.read_csv(file_path)
        list_of_dfs.append(df)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        continue

if list_of_dfs:
    gold = pd.concat(list_of_dfs, ignore_index=True)
else:
    print("No DataFrames could be successfully loaded.")


In [ ]:
gold.info()

In [ ]:
gold = gold[~gold.patient_summary.isnull()]

In [ ]:
gold = gold.sort_values(by='Unnamed: 0').reset_index()

In [ ]:
from sklearn.metrics import roc_auc_score

In [ ]:
roc_auc_score(gold.eligibility_result, predictions)

In [ ]:
from utils_102023 import eval_model

In [ ]:
eval_model(predictions, gold.eligibility_result, graph=True)

In [ ]:
pruned_set = gold[predictions >= 0.5]

In [ ]:
#pruned_set = pruned_set.groupby('patient_summary').head(20)


In [ ]:
# median number of results returned after trial checker
pruned_set.groupby('patient_summary').size().median()

In [ ]:
# mean number of results returned after trial checker
pruned_set.groupby('patient_summary').size().mean()

In [ ]:
pruned_set.patient_summary.nunique()

In [ ]:
pruned_set.this_space.nunique()

In [ ]:

def average_precision(label_array):
    total_yes = np.sum(label_array)
    if total_yes > 0:
        yes_indices = np.where(label_array == 1)[0] + 1
        precisions = []
        for index in yes_indices:
            precision = np.sum(label_array[0:index])/index
            precisions.append(precision)
        precisions = np.sum(np.array(precisions))
        return precisions / total_yes
    else:
        return 0

print(pruned_set.info())

print(pruned_set.eligibility_result.value_counts()/pruned_set.shape[0])
temp = pruned_set.groupby('patient_summary').head(20)
print(temp.eligibility_result.value_counts()/temp.shape[0])
temp = pruned_set.groupby('patient_summary').eligibility_result.apply(average_precision)
mapk = np.sum(temp)/len(temp)
print('map @ 20 of the round1.model on the external data with llama check')
print(mapk)